<a href="https://colab.research.google.com/github/devPartho3/Mern-Stack-Web-Dev-Marathon-/blob/main/TATA_Stock_Prediction_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
df=pd.read_csv("/content/NSE-TATAGLOBAL.csv")

In [ ]:
df.head()

,Date,Open,High,Low,Last,Close,Total Trade Quantity,Turnover (Lacs)
0,2018-09-28,234.05,235.95,230.20,233.50,233.75,3069914,7162.35
1,2018-09-27,234.55,236.80,231.10,233.80,233.25,5082859,11859.95
2,2018-09-26,240.00,240.00,232.50,235.00,234.25,2240909,5248.60
3,2018-09-25,233.30,236.75,232.00,236.25,236.10,2349368,5503.90
4,2018-09-24,233.55,239.20,230.75,234.00,233.30,3423509,7999.55


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Ensure Date is set as index or dropped before scaling
if 'Date' in df.columns:
    df_numeric = df.set_index('Date')
else:
    df_numeric = df.copy()

scaler = MinMaxScaler(feature_range=(0, 1))
df1 = scaler.fit_transform(df_numeric)

In [ ]:
df1.shape

(2035, 7)

In [ ]:
#creating a data structure with 60 timestamps and 1 output
x_train=[]
y_train=[]
for i in range(60,2035):
  x_train.append(df1[i-60:i,0])
  y_train.append(df1[i,0])

x_train,y_train=np.array(x_train),np.array(y_train)

In [ ]:
#reshaping
x_train=np.reshape(x_train,(x_train.shape[0],x_train.shape[1],1))

In [ ]:
x_train.shape

(1975, 60, 1)

In [ ]:
#building the RNN
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout

In [ ]:
input_shape=(x_train.shape[1],1)
model=Sequential()
model.add(LSTM(units=50,return_sequences=True,input_shape=input_shape))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [30]:
#Initializing the RNN
regressor = Sequential()
regressor.add(LSTM(units=50, return_sequences=True, input_shape=(x_train.shape[1], 1)))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [31]:
regressor.add(Dropout(0.2))

In [ ]:
#adding a second lstm layer and some dropout regularization
regressor.add(LSTM(units=50, return_sequences=True))
regressor.add(Dropout(0.2))

In [35]:
#adding a third lstm layer
regressor.add(LSTM(units=50, return_sequences=True))
regressor.add(Dropout(0.2))

ValueError: Input 0 with name 'None' of layer 'lstm_5' is incompatible with the layer: expected ndim=3, found ndim=2. Full shape received: (None, 50)

In [34]:
#adding last lstm layer
regressor.add(LSTM(units=50))
regressor.add(Dropout(0.2))

In [33]:
#adding the output layer
regressor.add(Dense(units=1))

#compiling the RNN
regressor.compile(optimizer='adam',loss='mean_squared_error')

#fitting the RNN to the training set
regressor.fit(x_train,y_train,epochs=10,batch_size=32)

Epoch 1/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - loss: 0.0458
Epoch 2/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0375
Epoch 3/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0372
Epoch 4/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0369
Epoch 5/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0370
Epoch 6/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - loss: 0.0366
Epoch 7/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0363
Epoch 8/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0367
Epoch 9/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0364
Epoch 10/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 0.0364


In [37]:
# To fix the error, we must scale our data using a scaler fitted only on the 1-feature 'Open' column.
# Let's fit a new scaler on the training part of the 'Open' column to match the dimensions.
open_scaler = MinMaxScaler(feature_range=(0, 1))
training_set = df.iloc[:2035, 1:2].values
open_scaler.fit(training_set)

# Making the predictions and preparing the inputs
dataset_train = df.iloc[:2035, 1:2]
dataset_test = df.iloc[2035:, 1:2]
dataset_total = pd.concat((dataset_train, dataset_test), axis=0)
real_stock_price = dataset_total[len(dataset_total) - len(dataset_test) - 60:].values
inputs = real_stock_price
inputs = inputs.reshape(-1, 1)
inputs = open_scaler.transform(inputs)

In [38]:
real_stock_price

array([[131.9 ],
       [134.5 ],
       [134.5 ],
       [134.75],
       [130.1 ],
       [128.  ],
       [125.  ],
       [124.4 ],
       [121.3 ],
       [122.3 ],
       [123.3 ],
       [124.6 ],
       [126.35],
       [123.45],
       [121.6 ],
       [123.  ],
       [124.55],
       [122.5 ],
       [122.8 ],
       [124.  ],
       [124.85],
       [126.15],
       [128.5 ],
       [127.7 ],
       [128.  ],
       [124.  ],
       [123.25],
       [122.5 ],
       [124.2 ],
       [121.65],
       [117.  ],
       [118.8 ],
       [119.  ],
       [118.2 ],
       [116.  ],
       [118.45],
       [114.  ],
       [112.  ],
       [110.  ],
       [110.15],
       [111.9 ],
       [115.95],
       [113.4 ],
       [113.  ],
       [113.75],
       [112.9 ],
       [112.  ],
       [114.1 ],
       [117.  ],
       [118.5 ],
       [118.3 ],
       [117.3 ],
       [116.5 ],
       [112.5 ],
       [118.  ],
       [117.6 ],
       [120.1 ],
       [121.8 ],
       [120.3 

In [47]:
inputs = inputs.reshape(-1, 1)
inputs = open_scaler.transform(inputs)

# Since we only have 60 total inputs, we create a single test sample of 60 timesteps
x_test = []
x_test.append(inputs[0:60, 0])
x_test = np.array(x_test)

# Reshape to match the expected 3D format for the LSTM (batch_size, timesteps, features)
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))